In [1]:
import numpy as np
import numpy.lib.recfunctions as recfun
from pathlib import Path
import matplotlib.pyplot as plt

import os

import time

import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
print("Testing IO for meshes ...")
pcd = o3d.io.read_point_cloud("../data/Challenge-ABC/Train/ply/0037.ply")
labels = np.loadtxt("../data/Challenge-ABC/Train/lb/0037.lb", dtype=int)
print(pcd)
#o3d.io.write_triangle_mesh("", pcd)

print(np.asarray(pcd.points))
print(np.asarray(pcd.points).shape[0])
print(len(pcd.points))
#o3d.visualization.draw_geometries([pcd])#,
#                                   zoom=0.3412,
#                                   front=[0.4257, -0.2125, -0.8795],
#                                   lookat=[2.6172, 2.0475, 1.532],
#                                   up=[-0.0694, -0.9768, 0.2024])
o3d.visualization.draw_plotly([pcd])

Testing IO for meshes ...
PointCloud with 20228 points.
[[15.          0.          0.        ]
 [15.          0.         22.        ]
 [14.99240017  0.47591901  0.        ]
 ...
 [13.76749992 -5.95447016  6.54984999]
 [13.5376997   6.45983982 10.1821003 ]
 [14.69849968 -2.99214005  7.19478989]]
20228
20228


In [3]:
pcd = o3d.io.read_point_cloud("../data/Challenge-ABC/Train/ply/0037.ply")
labels = np.loadtxt("../data/Challenge-ABC/Train/lb/0037.lb", dtype='int')
color_map = np.array([  
    [0.0, 0.0, 1.0],
    [1.0, 0.0, 0.0]   
])
# map labels to colors using the labels as indices
point_colors = color_map[labels]

# Cast the numpy array to Open3D's vector format and assign it
pcd.colors = o3d.utility.Vector3dVector(point_colors)
o3d.visualization.draw_plotly([pcd])

In [4]:
# down sampling test
downsampled_pcd = pcd.voxel_down_sample(voxel_size=0.75)
o3d.visualization.draw_plotly([downsampled_pcd])

In [5]:
def relative_voxel_downsample(pcd, resolution_percentage=0.02):
    """
    Downsamples a point cloud relative to its own bounding box size.
    resolution_percentage: The size of the voxel relative to the model's longest side.
                           (e.g., 0.02 means the voxel is 2% of the model's max dimension)
    """
    # get the absolute bounds of the current model
    min_bound = pcd.get_min_bound()
    max_bound = pcd.get_max_bound()
    
    # find the largest dimension 
    max_dim = np.max(max_bound - min_bound)
    
    # calculate the absolute voxel size needed for this specific model
    dynamic_voxel_size = max_dim * resolution_percentage
    
    # apply down sampling
    downsampled_pcd = pcd.voxel_down_sample(voxel_size=dynamic_voxel_size)
    
    return downsampled_pcd

In [6]:
rel_pcd = relative_voxel_downsample(pcd, resolution_percentage=0.05)
o3d.visualization.draw_plotly([rel_pcd])

In [7]:
def downsample_point_clouds(input_dir, output_dir, resolution=0.02):
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    
    output_path.mkdir(parents=True, exist_ok=True)

    ply_files = list(input_path.glob("*.ply"))
    total_files = len(ply_files)
    
    if total_files == 0:
        print(f"No .ply files found in {input_dir}")
        return

    print(f"Processing {total_files} files at {resolution*100}% (of max dim size) relative resolution...")
    print("-" * 75)

    for i, file_path in enumerate(ply_files, 1):
        pcd = o3d.io.read_point_cloud(str(file_path))
        
        if not pcd.has_points():
            print(f"[{i}/{total_files}] ERROR: Skipped {file_path.name} (Empty or unreadable)")
            continue
            
        original_count = len(pcd.points)
        downsampled_pcd = relative_voxel_downsample(pcd, resolution_percentage=resolution)
        out_file = output_path / file_path.name
        o3d.io.write_point_cloud(str(out_file), downsampled_pcd)
        
        print(f"[{i}/{total_files}] {file_path.name:<10} | Points: {original_count:<10} -> {len(downsampled_pcd.points)}")

downsample_point_clouds("../data/Challenge-ABC/Train/ply/", "../data/Challenge-ABC/Train/downsampled/", resolution=0.01)

Processing 198 files at 1.0% (of max dim size) relative resolution...
---------------------------------------------------------------------------
[1/198] 2777.ply   | Points: 2004       -> 1343
[2/198] 0746.ply   | Points: 12789      -> 11732
[3/198] 1452.ply   | Points: 118332     -> 21825
[4/198] 2295.ply   | Points: 10200      -> 9623
[5/198] 2046.ply   | Points: 17398      -> 16699
[6/198] 1898.ply   | Points: 17180      -> 16772
[7/198] 2678.ply   | Points: 24817      -> 24196
[8/198] 1139.ply   | Points: 14842      -> 13823
[9/198] 0156.ply   | Points: 14519      -> 13855
[10/198] 2425.ply   | Points: 6867       -> 5959
[11/198] 1405.ply   | Points: 16704      -> 16216
[12/198] 0654.ply   | Points: 12747      -> 9150
[13/198] 1961.ply   | Points: 27323      -> 26118
[14/198] 2304.ply   | Points: 15533      -> 15331
[15/198] 0037.ply   | Points: 20228      -> 19503
[16/198] 2354.ply   | Points: 7433       -> 5880
[17/198] 2438.ply   | Points: 4821       -> 4343
[18/198] 0039.ply  

In [8]:
def check_downsampled_points(original_file, downsampled_file):
    # read the two files and get the points data
    pcd_orig = o3d.io.read_point_cloud(original_file)
    pcd_down = o3d.io.read_point_cloud(downsampled_file)
    orig_pts = np.asarray(pcd_orig.points)
    down_pts = np.asarray(pcd_down.points)
    
    print(f"Checking {len(down_pts)} downsampled points against {len(orig_pts)} original points...")
    print("This is the brute-force O(N*M) method, it might take some time depending on model size...")
    print("-" * 60)
    
    exact_matches = 0
    new_points = 0
    
    # loop through every single point in the downsampled file
    for pt in down_pts:
        # compare current downsampled point [x, y, z] against ALL original points [x, y, z]
        # np.all(..., axis=1) ensures x, y, and z all match exactly on the same row
        # np.any(...) checks if at least one row returned True
        is_in_original = np.any(np.all(orig_pts == pt, axis=1))
        
        if is_in_original:
            exact_matches += 1
        else:
            new_points += 1

    print(f"Exact original points kept: {exact_matches}")
    print(f"Brand new points (centroids): {new_points}")
    print(f"Are they completely new? {'YES' if exact_matches == 0 else 'NO'}")

check_downsampled_points("../data/Challenge-ABC/Train/ply/2777.ply", "../data/Challenge-ABC/Train/downsampled/2777.ply")
print("\n")
check_downsampled_points("../data/Challenge-ABC/Train/ply/0037.ply", "../data/Challenge-ABC/Train/downsampled/0037.ply")
print("\n")
check_downsampled_points("../data/Challenge-ABC/Train/ply/1452.ply", "../data/Challenge-ABC/Train/downsampled/1452.ply")

Checking 1343 downsampled points against 2004 original points...
This is the brute-force O(N*M) method, it might take some time depending on model size...
------------------------------------------------------------
Exact original points kept: 782
Brand new points (centroids): 561
Are they completely new? NO


Checking 19503 downsampled points against 20228 original points...
This is the brute-force O(N*M) method, it might take some time depending on model size...
------------------------------------------------------------
Exact original points kept: 18809
Brand new points (centroids): 694
Are they completely new? NO


Checking 21825 downsampled points against 118332 original points...
This is the brute-force O(N*M) method, it might take some time depending on model size...
------------------------------------------------------------
Exact original points kept: 1200
Brand new points (centroids): 20625
Are they completely new? NO


In [9]:
print("======================================================================")
print("===Original Model===")
print("======================================================================")
print(pcd)
print(np.asarray(pcd.points))
o3d.visualization.draw_plotly([pcd])

print("======================================================================")
print("===Model downsampled with uniform downsapling with k=3 ===")
print("======================================================================")
uni_pcd =pcd.uniform_down_sample(every_k_points=3)
print(uni_pcd)
print(np.asarray(uni_pcd.points))
o3d.visualization.draw_plotly([uni_pcd])

print("======================================================================")
print("===Model downsampled with voxel downsapling - Open3d direct result===")
print("======================================================================")
vox_pcd = pcd.voxel_down_sample_and_trace(voxel_size=1, min_bound=pcd.get_min_bound(), max_bound=pcd.get_max_bound())
print(vox_pcd[0])
print(np.asarray(vox_pcd[0].points))
o3d.visualization.draw_plotly([vox_pcd[0]])


#print(vox_pcd[1])
print("Original indices of the points in each voxel bucket:")
print(vox_pcd[2])


print("======================================================================")
print("===Model downsampled with voxel downsapling but with taking the first point from each voxel => Original indexes ===")
print("======================================================================")
# extract the trace list 
trace_list = vox_pcd[2]

# grab the very first original index from each populated voxel bucket
downsampled_original_indices = np.array([voxel_indices[0] for voxel_indices in trace_list])

# convert to a clean NumPy array, ensure uniqueness, and sort them
downsampled_original_indices = np.sort(downsampled_original_indices)
print(downsampled_original_indices)
print("Are there any duplicates?")
is_unique = downsampled_original_indices.size == np.unique(downsampled_original_indices).size
print(f"{"NO" if is_unique else "YES"}")

original_pcd_downsapled = pcd.select_by_index(downsampled_original_indices)
print(original_pcd_downsapled)
o3d.visualization.draw_plotly([original_pcd_downsapled])

orig_pts = np.asarray(pcd.points)
down_pts = np.asarray(original_pcd_downsapled.points)
print(f"Checking {len(down_pts)} downsampled points against {len(orig_pts)} original points...")

exact_matches = 0
new_points = 0
    
# loop through every single point in the downsampled file
for pt in down_pts:

    is_in_original = np.any(np.all(orig_pts == pt, axis=1))
        
    if is_in_original:
        exact_matches += 1
    else:
        new_points += 1
print(f"Exact original points kept: {exact_matches}")
print(f"Brand new points (centroids): {new_points}")


print("======================================================================")
print("===Model downsampled with voxel downsapling but with taking a random point from each voxel => Original indexes ===")
print("======================================================================")
# extract the trace list 
trace_list = vox_pcd[2]

# grab the very first original index from each populated voxel bucket
downsampled_original_indices = np.array([voxel_indices[np.random.randint(0, len(voxel_indices))] for voxel_indices in trace_list])

# convert to a clean NumPy array, ensure uniqueness, and sort them
downsampled_original_indices = np.sort(downsampled_original_indices)
print(downsampled_original_indices)
print("Are there any duplicates?")
is_unique = downsampled_original_indices.size == np.unique(downsampled_original_indices).size
print(f"{"NO" if is_unique else "YES"}")



original_pcd_downsapled = pcd.select_by_index(downsampled_original_indices)
print(original_pcd_downsapled)
o3d.visualization.draw_plotly([original_pcd_downsapled])

orig_pts = np.asarray(pcd.points)
down_pts = np.asarray(original_pcd_downsapled.points)
print(f"Checking {len(down_pts)} downsampled points against {len(orig_pts)} original points...")

exact_matches = 0
new_points = 0
    
# loop through every single point in the downsampled file
for pt in down_pts:

    is_in_original = np.any(np.all(orig_pts == pt, axis=1))
        
    if is_in_original:
        exact_matches += 1
    else:
        new_points += 1
print(f"Exact original points kept: {exact_matches}")
print(f"Brand new points (centroids): {new_points}")

===Original Model===
PointCloud with 20228 points.
[[15.          0.          0.        ]
 [15.          0.         22.        ]
 [14.99240017  0.47591901  0.        ]
 ...
 [13.76749992 -5.95447016  6.54984999]
 [13.5376997   6.45983982 10.1821003 ]
 [14.69849968 -2.99214005  7.19478989]]


===Model downsampled with uniform downsapling with k=3 ===
PointCloud with 6743 points.
[[15.          0.          0.        ]
 [14.9698      0.95135897  0.        ]
 [14.81159973  2.37001991  0.        ]
 ...
 [13.32789993  6.88239002 15.42879963]
 [13.09739971 -7.31159019  7.18394995]
 [13.5376997   6.45983982 10.1821003 ]]


===Model downsampled with voxel downsapling - Open3d direct result===
PointCloud with 3816 points.
[[14.04640007 -5.26295996 11.48910046]
 [13.9071002  -5.61680984 10.48760033]
 [11.68703334 -9.39459324 11.52606678]
 ...
 [ 9.52246434 -7.47933435  0.        ]
 [-1.55729002 14.91654992 16.46815014]
 [14.6012667   3.42298166 15.60603348]]


Original indices of the points in each voxel bucket:
[IntVector[20137], IntVector[20134, 20136], IntVector[20115, 20117, 20119], IntVector[20113], IntVector[20108, 20110, 20112], IntVector[20088, 20090], IntVector[20082], IntVector[20078, 20080], IntVector[20066, 20068], IntVector[20062, 20064], IntVector[20056, 20058, 20060], IntVector[20052, 20054], IntVector[20048, 20050], IntVector[20026], IntVector[20014], IntVector[20010, 20012], IntVector[20002, 20004, 20006], IntVector[19998, 20000], IntVector[19996], IntVector[19992, 19994], IntVector[19988, 19990], IntVector[19986], IntVector[19984], IntVector[19970, 19972], IntVector[19966, 19968], IntVector[19962, 19964], IntVector[19950, 19952], IntVector[19946, 19948], IntVector[19932, 19934], IntVector[19922, 19924], IntVector[19915], IntVector[19904, 19906, 19908], IntVector[19890, 19892], IntVector[19880, 19882], IntVector[19876, 19878], IntVector[19862], IntVector[19858, 19860], IntVector[19850, 19852], IntVector[19846, 19848], IntVec

Checking 3816 downsampled points against 20228 original points...
Exact original points kept: 3816
Brand new points (centroids): 0
===Model downsampled with voxel downsapling but with taking a random point from each voxel => Original indexes ===
[    1    12    17 ... 20216 20219 20224]
Are there any duplicates?
NO
PointCloud with 3816 points.


Checking 3816 downsampled points against 20228 original points...
Exact original points kept: 3816
Brand new points (centroids): 0


In [10]:
# generate a global index array to keep track of the original points order
global_indexes = np.arange(len(pcd.points))

# create indexes masks fpr the 2 classes
edge_mask = []
non_edge_mask = []
for i in range(len(labels)):
    if labels[i] == 1:
        edge_mask.append(i)
    else:
        non_edge_mask.append(i)
print(edge_mask)
print(non_edge_mask)

# isolate the class 0 points for open3d
non_edge_pcd = pcd.select_by_index(non_edge_mask)
# the same for the indexes
non_edge_global_indices = global_indexes[non_edge_mask]
print(non_edge_global_indices)

# perform downsampling on the non edge points
non_edge_downsampled_tuple = non_edge_pcd.voxel_down_sample_and_trace(voxel_size=1, min_bound=non_edge_pcd.get_min_bound(), max_bound=non_edge_pcd.get_max_bound())
non_edge_downsampled_pcd_trace_list = non_edge_downsampled_tuple[2]

downsampled_voxel_indexes = np.array([voxel_indexes[np.random.randint(0, len(voxel_indexes))] for voxel_indexes in non_edge_downsampled_pcd_trace_list])
downsampled_voxel_indexes = np.sort(downsampled_voxel_indexes)

# get back the global indexes
selected_non_edge_global_indices = non_edge_global_indices[downsampled_voxel_indexes]

# intermediate visualization
inter_pcd = pcd.select_by_index(selected_non_edge_global_indices)
o3d.visualization.draw_plotly([inter_pcd])

# final reintegration:
# 1- get the global indexes of all the edges/class 1
edge_global_indices = global_indexes[edge_mask]
# 2- merge them together and sort (TODO : maybe the sorting isn't necessary?)
final_global_indices = np.concatenate([edge_global_indices, selected_non_edge_global_indices])
final_global_indices = np.sort(final_global_indices)

# visualisztion test
res_pcd = pcd.select_by_index(final_global_indices)
o3d.visualization.draw_plotly([res_pcd])

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

In [11]:
pcd_fps = pcd.farthest_point_down_sample(5000)
o3d.visualization.draw_plotly([pcd_fps])

# this method only returns a pcd so we'll use a KDTree to get back the original points
pcd_kdtree = o3d.geometry.KDTreeFlann(pcd)
# for each point in the fps kdtree find the nearest neighbour in the original pcd
fps_original_idx = []
for point in pcd_fps.points:
    fps_original_idx.append(pcd_kdtree.search_knn_vector_3d(point, 1)[1][0])

print(pcd_fps)
print(fps_original_idx)

pcd_fps_from_original = pcd.select_by_index(fps_original_idx)
o3d.visualization.draw_plotly([pcd_fps_from_original])

PointCloud with 5000 points.
[0, 1, 3, 5, 7, 11, 13, 15, 17, 19, 22, 24, 28, 30, 32, 37, 39, 41, 43, 45, 47, 49, 54, 59, 64, 66, 68, 70, 72, 75, 77, 79, 81, 83, 85, 91, 93, 95, 97, 102, 105, 108, 110, 114, 117, 121, 124, 127, 130, 132, 136, 138, 142, 144, 146, 148, 150, 154, 157, 160, 162, 167, 175, 178, 185, 188, 192, 196, 200, 202, 206, 209, 211, 213, 216, 218, 221, 224, 226, 228, 230, 236, 239, 242, 247, 250, 254, 256, 258, 260, 262, 265, 268, 271, 274, 278, 280, 282, 284, 292, 297, 301, 305, 308, 310, 312, 315, 318, 323, 325, 327, 329, 331, 333, 335, 337, 341, 343, 345, 347, 349, 351, 354, 358, 361, 363, 365, 368, 370, 372, 374, 377, 381, 383, 387, 392, 397, 403, 407, 410, 419, 425, 427, 429, 433, 436, 440, 442, 457, 458, 459, 460, 463, 468, 470, 471, 472, 476, 477, 479, 481, 483, 488, 489, 490, 500, 501, 503, 510, 518, 528, 533, 541, 550, 554, 558, 563, 566, 567, 573, 578, 580, 582, 586, 589, 594, 597, 609, 618, 624, 630, 645, 647, 653, 657, 661, 665, 670, 678, 682, 686, 687, 688,

In [15]:
# generate a global index array to keep track of the original points order
global_indexes = np.arange(len(pcd.points))

# create indexes masks fpr the 2 classes
edge_mask = []
non_edge_mask = []
for i in range(len(labels)):
    if labels[i] == 1:
        edge_mask.append(i)
    else:
        non_edge_mask.append(i)
print(edge_mask)
print(non_edge_mask)

# isolate the class 0 points for open3d
non_edge_pcd = pcd.select_by_index(non_edge_mask)
# the same for the indexes
non_edge_global_indices = global_indexes[non_edge_mask]
print(non_edge_global_indices)

# perform downsampling on the non edge points with 5000 points
non_edge_downsampled_fps_pcd = non_edge_pcd.farthest_point_down_sample(5000)
pcd_kdtree = o3d.geometry.KDTreeFlann(non_edge_pcd)
non_edge_relative_indices = []
for point in non_edge_downsampled_fps_pcd.points:
    non_edge_relative_indices.append(pcd_kdtree.search_knn_vector_3d(point, 1)[1][0])

# intermediate visualization
selected_non_edge_global_indices = non_edge_global_indices[non_edge_relative_indices]

inter_pcd = pcd.select_by_index(selected_non_edge_global_indices)
o3d.visualization.draw_plotly([inter_pcd])

# final reintegration:
# 1- get the global indexes of all the edges/class 1
edge_global_indices = global_indexes[edge_mask]
# 2- merge them together and sort (TODO : maybe the sorting isn't necessary?)
final_global_indices = np.concatenate([edge_global_indices, selected_non_edge_global_indices])
final_global_indices = np.sort(final_global_indices)

# visualisztion test
res_pcd = pcd.select_by_index(final_global_indices)
o3d.visualization.draw_plotly([res_pcd])

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,